# Vigade analüüs — Õisi kursuste soovitaja

Kodutöö 1. ülesanne: vahesammude vigade analüüs.

Analüüsitavad vahesammud:
1. **Metaandmete filtreerimine** — kas filtrid jätsid liiga vähe/palju kursusi alles?
2. **RAG vektorotsing** — kas semantiline otsing leidis valed ained?
3. **LLM vastuse genereerimine** — kas LLM hallutsineeris või vastas valesti?

In [6]:
import pandas as pd

## 1. Andmete laadimine

In [2]:
df = pd.read_csv('tagasiside_log.csv')
df.head()

,Aeg,Kasutaja päring,Filtrid,Leitud ID-d,Leitud ained,LLM Vastus,Hinnang,Veatüüp
0,2026-03-01 16:37:48,tahan sporti teha,semester: kevad; asukoht: Tartu; hindamine: Er...,"['KKSP.00.027', 'KKSB.01.034', 'KKSP.03.054', ...","['Spordi terviseks II', 'Tervislik liikumishar...","\nKui soovid sporti teha, siis **Spordi tervis...",👍 Hea,NaN
1,2026-03-01 16:39:37,samuti soovin ainet kus saan õppida eesti keelt,pole oluline,"['HVLC.09.001', 'HVLC.09.065', 'HVLC.09.060', ...","['Eesti keel arstiteaduse tudengitele I, tase ...",\n\nEesti keele õppimiseks on mitmeid valikuid...,👍 Hea,NaN
2,2026-03-01 16:40:37,soovin kursust kus saan tegeleda andmeteaduseg...,pole oluline,"['SVNC.00.228', 'LTAT.02.002', 'SVMJ.04.015', ...","['Sissejuhatus andmeteadusse', 'Sissejuhatus a...",\n\nSel juhul on sinule sobivaim **Sissejuhatu...,👍 Hea,NaN
3,2026-03-01 16:41:25,soovin ainet kus saan laulda,pole oluline,"['P2VK.02.728', 'MV00.00.018', 'MV00.00.017', ...","['Improvisatsioon', 'Biomeedikumi segakoor II'...",Laulmise võimaluseks on sinule **Biomeedikumi ...,👍 Hea,NaN
4,2026-03-01 16:43:04,"soovin ainet, kus saan õppida informaatikat, a...",pole oluline,"['LTAT.03.002', 'MTAT.03.312', 'MTAT.03.206', ...","['Sissejuhatus erialasse', 'Informaatika kooli...",\n\nHea valik on **Digitaalse maailma alused**...,👎 Halb,Otsing leidis valed ained (RAG viga)


## 2. Ülevaade — kõik päringud

In [3]:
kokku = len(df)
head = (df['Hinnang'] == '👍 Hea').sum()
halb = (df['Hinnang'] == '👎 Halb').sum()

print(f'Päringuid kokku:  {kokku}')
print(f'Heade vastuste arv:  {head} ({head/kokku:.0%})')
print(f'Halbade vastuste arv: {halb} ({halb/kokku:.0%})')

Päringuid kokku:  21
Heade vastuste arv:  12 (57%)
Halbade vastuste arv: 9 (43%)


## 3. Halvad juhtumid

In [4]:
halvad = df[df['Hinnang'] == '👎 Halb'].copy()
halvad[['Aeg', 'Kasutaja päring', 'Filtrid', 'Leitud ained', 'Veatüüp']]

,Aeg,Kasutaja päring,Filtrid,Leitud ained,Veatüüp
4,2026-03-01 16:43:04,"soovin ainet, kus saan õppida informaatikat, a...",pole oluline,"['Sissejuhatus erialasse', 'Informaatika kooli...",Otsing leidis valed ained (RAG viga)
5,2026-03-01 17:11:03,soovin kursust kus saan keraamikaga ja kunstig...,pole oluline,"['Artisti karjääri arendamine', 'Origami ja ma...",Otsing leidis valed ained (RAG viga)
6,2026-03-01 17:11:52,soovin ainet kus saan õppida oraatorioskusi,pole oluline,"['Retoorika', 'Ukraina keel algajatele II (ing...",Otsing leidis valed ained (RAG viga)
8,2026-03-01 17:13:47,tahan rohkem kogemust tarkvaraarenduses,pole oluline,"['Sissejuhatus tarkvaraettevõtlusse', ""Hackath...",LLM hallutsineeris/vastas valesti
9,2026-03-01 17:14:44,"mul on vähe teadmisi majandusvaldkonnast, taha...",pole oluline,"['Majanduse põhialused', 'Ettevõtlusega alusta...",Otsing leidis valed ained (RAG viga)
11,2026-03-01 17:39:07,?,pole oluline,['Vene keel Vene filmide baasil. Vestluskursus...,Otsing leidis valed ained (RAG viga)
14,2026-03-01 17:42:16,"soovin baasi arstinduses, aga ei oma meditsiin...",pole oluline,"['Sissejuhatus erialaõpingutesse', 'Patofüsiol...",Filtrid olid liiga karmid/valed
15,2026-03-01 17:42:49,tahan ratsutada,pole oluline,"['Ratastoolioskuste alused', 'Ajakirjanik ja v...",LLM hallutsineeris/vastas valesti
20,2026-03-01 17:50:59,"kui ma olen informaatika tudeng, siis millisei...",pole oluline,"['Sissejuhatus erialasse', 'Arvutitehnoloogia'...",LLM hallutsineeris/vastas valesti


## 4. Vigade analüüsi tabel vahesammude kaupa

In [5]:
vahesammud = {
    'Metaandmete filtreerimine': 'Filtrid olid liiga karmid/valed',
    'RAG vektorotsing':          'Otsing leidis valed ained (RAG viga)',
    'LLM vastuse genereerimine': 'LLM hallutsineeris/vastas valesti',
}

read = []
for samm, veatyyp in vahesammud.items():
    arv = (halvad['Veatüüp'] == veatyyp).sum()
    read.append({
        'Vahesamm': samm,
        'Vigade arv': arv,
        '% kõigist vigastest päringutes': f'{arv/halb:.1%}' if halb > 0 else '0%',
        '% kõigist päringutes': f'{arv/kokku:.1%}',
    })

analyys = pd.DataFrame(read)
analyys

,Vahesamm,Vigade arv,% kõigist vigastest päringutes,% kõigist päringutes
0,Metaandmete filtreerimine,1,11.1%,4.8%
1,RAG vektorotsing,5,55.6%,23.8%
2,LLM vastuse genereerimine,3,33.3%,14.3%


## 5. Järeldused

Testiti 21 päringut. 12 vastust (57%) hinnati heaks, 9 vastust (43%) halvaks.

**Kõige sagedasem viga: RAG vektorotsing** (5/9 veast, 56%)

RAG otsing on selgelt nõrgim koht rakenduses. Probleem ilmneb eriti kahel juhul:
- **Liiga kitsas teema** mida andmebaasis napilt katab — nt "keraamika" ja "ratsutamine". Andmebaasis pole selliseid aineid, kuid RAG tagastab ikkagi 5 tulemust mis semantiliselt on kaugel (keraamika asemel: artisti karjäär, origami, alkeemia; ratsutamise asemel: ratastoolioskused, robootika).
- **Liiga lai või ebamäärane päring** — nt "?" tagastas täiesti juhusliku tulemuse (vene keel filmidest, rütmika). Süsteem peaks sellised päringud tagasi lükkama.
- **Sihtrühma mõistmine** — "informaatika, aga ei ole informaatik" tagastas informaatika erialastudengite ained, mitte sissejuhatavaid aineid mitteinformaatikutele.

**LLM hallutsinatsioonid** (3/9 veast, 33%)

LLM ignoreerib kontekstis olevaid aineid ja leiutab oma treeningandmetest aineid mida andmebaasis pole (nt "Tarkvaraarenduse põhitõed", "Objektorienteeritud programmeerimine", "Digitaalse maailma alused"). Probleem on eriti tugev kui RAG tulemused on nõrgad — LLM täidab "tühiku" ise. Praegune prompt ("KEELATUD: Ära nimeta ühtegi konkreetset kursust") vähendab seda, aga ei kõrvalda täielikult.

**Filtreerimise vead** (1/9 veast, 11%)

Filtrid ise töötasid enamasti hästi — vaid üks juhtum kus "arstindus ilma meditsiinilise taustata" tagastas kõrgtaseme meditsiiniained (Patofüsioloogia II, Sõja- ja katastroofimeditsiin). Tegemist on pigem RAG probleemiga: otsing ei arvesta eeldusaineid ega õppetaset.

**Võimalikud parandused:**
- RAG: lisada embeddingusse ka `eeldusained` ja `oppeaste` info, et otsing arvestaks raskusastet.
- RAG: lisada tulemuse usalduslävi — kui maksimaalne sarnasuskoor on alla mingi piiri (nt 0.4), kuvada hoiatus "Sinu päringule vastavaid aineid ei leitud".
- LLM: kaaluda LLM täielikku eemaldamist kommentaaride genereerimisest, kuna hallutsinatsioonirisk on kõrge ja kastid annavad piisava info.